# Ground-truth tangent recovery from a learned score

**Goal:** Test whether our score-Jacobian estimator produces the **true tangent subspace**, rather than merely scoring well on MNIST handwriting-direction proxies. This notebook is standalone; it does **not** rerun your 500-image MNIST calibration.

We use a uniformly sampled circle embedded in ambient **8-dimensional pixel-like space**. Its true intrinsic dimension is **1** and its tangent projector is known analytically. We evaluate the same normalized score-Jacobian idea on an **exact Gaussian-smoothed oracle score** and on a **noise-conditioned neural score** trained only on synthetic data.

- **No classifier, no surrogate, no adversarial-query access.**
- Three Gaussian noise standard deviations: `0.030`, `0.128`, `0.342`, numerically comparable to your MNIST experiment. They are **not DDPM timesteps** and need not share the same interpretation on this synthetic distribution.
- Split: 40 points for choosing a proxy-ranked noise scale, 100 untouched held-out points for **true tangent-projector error**.
- First we assume the **known true dimension d=1** when extracting one eigenvector. This isolates tangent estimation from dimension estimation. Do not interpret this as validating unknown intrinsic dimension.

At the end, the notebook prints a mean error table, tests whether fit-only proxy ranking selects the true best scale, shows confidence intervals, and saves results and model weights.

In [ ]:
import os, random, time, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import ive
from scipy.stats import spearmanr
import torch
from torch import nn
from IPython.display import display

SEED=73
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
D=8; R=1.0
SIGMAS=[0.030,0.128,0.342]
N_FIT=40; N_TEST=100
TRAIN_STEPS=2500  # start here; increase if actual learned-score errors remain high
BATCH=512
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SAVE_TO_DRIVE=True  # strong recommendation given previous Colab disconnections

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT=Path('/content/drive/MyDrive/synthetic_tangent_ground_truth')
    except ImportError:
        OUT=Path.cwd()/'synthetic_tangent_ground_truth'
else:
    OUT=Path('/content/synthetic_tangent_ground_truth') if Path('/content').exists() else Path.cwd()/'synthetic_tangent_ground_truth'
OUT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| Output:', OUT)

## 1 — Define a curved manifold with exact tangent ground truth

Fix two orthonormal vectors \(b_1,b_2\in\mathbb R^8\). For a point on the circle,
\[
x(\theta)=\cos\theta\,b_1+\sin\theta\,b_2,
\qquad
u(\theta)=-\sin\theta\,b_1+\cos\theta\,b_2.
\]
The true rank-one tangent projector is \(P_T(\theta)=u(\theta)u(\theta)^\top\). The manifold is **curved** even though its containing two-plane is linear.

We compute \(Q_\sigma=-\sigma^2\operatorname{sym}(J_s)\). For a Gaussian-smoothed **uniform circle**, an exact oracle is available using modified Bessel functions. At these noise scales and clean circle points, the eigenvector of \(Q_\sigma\) with the smallest eigenvalue is the oracle tangent direction.

In [ ]:
base=np.random.default_rng(SEED).normal(size=(D,2)); B,_=np.linalg.qr(base)
P_PLANE=B@B.T

def make_points(theta):
    theta=np.asarray(theta)
    x= R*(np.cos(theta)[:,None]*B[:,0] + np.sin(theta)[:,None]*B[:,1])
    u= -np.sin(theta)[:,None]*B[:,0] + np.cos(theta)[:,None]*B[:,1]
    return x,u

def oracle_Q(x,sigma):
    # Exact normalized score Jacobian of a uniformly sampled circle convolved with Gaussian noise.
    z=x @ B
    r=np.maximum(np.linalg.norm(z,axis=1),1e-12)
    e=(z/r[:,None])@B.T
    k=R*r/(sigma*sigma)
    A=ive(1,k)/ive(0,k)
    Aprime = 1-A/k-A*A
    P_rad=e[:,:,None]*e[:,None,:]
    J_mu=(R*R/sigma**2)*Aprime[:,None,None]*P_rad + (R*A/r)[:,None,None]*(P_PLANE[None,:,:]-P_rad)
    return np.eye(D)[None,:,:]-J_mu

def diagnostics(Q,u,x,theta,sigma,rng):
    lam,V=np.linalg.eigh(0.5*(Q+Q.transpose(0,2,1)))
    uhat=V[:,:,0]
    cos2=np.sum(uhat*u,axis=1)**2
    err=np.sqrt(np.maximum(0,2*(1-np.minimum(1,cos2))))
    defect=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
    deltas=np.array([-0.14,-0.08,0.08,0.14])
    neighbors=[]
    for delta in deltas:
        xd,_=make_points(theta+delta)
        v=xd-x
        neighbors.append(v/(np.linalg.norm(v,axis=1,keepdims=True)+1e-12))
    nb=np.stack(neighbors,axis=1)
    rand=rng.normal(size=(len(x),16,D))
    rand=rand/(np.linalg.norm(rand,axis=2,keepdims=True)+1e-12)
    nb_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,nb)**2,axis=-1),axis=1)
    rand_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,rand)**2,axis=-1),axis=1)
    proxy=np.minimum(1,nb_energy/(rand_energy+1e-12))
    soft_dim=np.sum(1-np.clip(lam,0,1),axis=1)
    return err, defect, proxy, soft_dim, lam

## 2 — Oracle sanity check (no neural model)

For each \(\sigma\), diagonalize the exact \(Q_\sigma\), take the **smallest-eigenvalue eigenvector**, and calculate
\[
\hat P_T=\hat u\hat u^\top,
\qquad
 e_T=\frac{\|\hat P_T-P_T\|_F}{\|P_T\|_F}
 =\sqrt{2(1-(\hat u^\top u)^2)}.
\]
For this 1-dimensional tangent, \(0\le e_T\le\sqrt2\). A value of 0 means perfect tangent alignment; \(\sqrt2\) means orthogonal.

We also measure a **data-only proxy**, defined as the score-operator response to short **one-sided chords** between neighboring circle samples divided by its response to 16 random directions. This proxy does **not** use the known tangent projector. It is intentionally imperfect, like the MNIST neighbor-direction proxy.

In [ ]:
th_fit=np.random.default_rng(SEED+1).uniform(0,2*np.pi,N_FIT)
th_test=np.random.default_rng(SEED+2).uniform(0,2*np.pi,N_TEST)

# Oracle experiment
records=[]
for split,theta in [('fit',th_fit),('test',th_test)]:
    x,u=make_points(theta)
    for sigma in SIGMAS:
        Q=oracle_Q(x,sigma)
        err,defect,proxy,soft_dim,_=diagnostics(Q,u,x,theta,sigma,np.random.default_rng(SEED+13))
        for i in range(len(x)):
            records.append({'model':'exact_score','split':split,'index':i,'sigma':sigma,'tangent_error':err[i],'projector_defect':defect[i],'neighbor_proxy':proxy[i],'soft_dimension':soft_dim[i]})

In [ ]:
oracle=pd.DataFrame(records)
display(oracle[oracle.split=='test'].groupby('sigma')[['tangent_error','neighbor_proxy','projector_defect','soft_dimension']].mean().round(6))
assert oracle['tangent_error'].max()<1e-5, 'Exact-score geometry sanity check failed'
print('PASS: exact score recovers the tangent space at all three scales.')

## 3 — Train a noise-conditioned score model (synthetic DSM)

Train \(\epsilon_\theta(y,\sigma)\) to predict Gaussian noise from \(y=x+\sigma\epsilon\), for \(x\) sampled uniformly on the circle and \(\epsilon\sim\mathcal N(0,I)\).

\[
\mathcal L(\theta)=\mathbb E_{x,\sigma,\epsilon}\|\epsilon_\theta(x+\sigma\epsilon,\sigma)-\epsilon\|_2^2,
\qquad s_\theta(y,\sigma)=-\epsilon_\theta(y,\sigma)/\sigma.
\]
This trains **only a generative score model**, never a classifier. A checkpoint is saved in Drive and reused on reruns.

In [ ]:
class EpsNetwork(nn.Module):
    def __init__(self,D=8,width=128):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(D+1,width), nn.SiLU(),
            nn.Linear(width,width), nn.SiLU(),
            nn.Linear(width,width), nn.SiLU(),
            nn.Linear(width,D))
    def forward(self,y,sigma):
        if not isinstance(sigma,torch.Tensor):
            sigma=torch.full((len(y),1),float(sigma),device=y.device,dtype=y.dtype)
        if sigma.ndim==1:
            sigma=sigma[:,None]
        return self.net(torch.cat([y,torch.log(sigma)],dim=1))

model=EpsNetwork(D).to(DEVICE)
checkpoint=OUT/'synthetic_eps_network.pt'
if checkpoint.exists():
    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE,weights_only=True))
    print('Loaded existing model checkpoint from',checkpoint)
else:
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    B_t=torch.tensor(B,dtype=torch.float32,device=DEVICE)
    sig_t=torch.tensor(SIGMAS,device=DEVICE,dtype=torch.float32)
    start=time.time()
    for it in range(TRAIN_STEPS):
        idx=torch.randint(len(SIGMAS),(BATCH,),device=DEVICE)
        sigma=sig_t[idx,None]
        angle=2*np.pi*torch.rand(BATCH,device=DEVICE)
        c=torch.stack([torch.cos(angle),torch.sin(angle)],dim=1)
        clean=c@B_t.T
        noise=torch.randn_like(clean)
        y=clean+sigma*noise
        predicted=model(y,sigma)
        loss=(predicted-noise).square().mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (it+1)%500==0 or it==0:
            print(f'step {it+1}/{TRAIN_STEPS} | DSM loss {loss.item():.4f} | elapsed {time.time()-start:.1f}s')
    torch.save(model.state_dict(),checkpoint)
    print('Saved checkpoint:',checkpoint)
model.eval()

## 4 — Calculate learned score Jacobians and true tangent errors

For the epsilon-prediction model:
\[
Q_\sigma=-\sigma^2J_s=\sigma J_{\epsilon_\theta}.
\]
We estimate this Jacobian by batched central finite differences in 8 dimensions. This is cheap in the synthetic setting and avoids differentiating through PyTorch attention or computing full Hessians. We then **symmetrize** and extract its smallest-eigenvalue eigenvector (using the known \(d=1\)).

In [ ]:
@torch.no_grad()
def learned_Q(x,sigma,h=0.001):
    # Stable batched finite differences; no Hessian or second backward required.
    X=np.asarray(x,dtype=np.float32)
    out=[]
    for j in range(D):
        plus=X.copy(); minus=X.copy(); plus[:,j]+=h;minus[:,j]-=h
        a=model(torch.from_numpy(plus).to(DEVICE),sigma).cpu().numpy()
        b=model(torch.from_numpy(minus).to(DEVICE),sigma).cpu().numpy()
        out.append((a-b)/(2*h))
    J=np.stack(out,axis=-1)
    Q=sigma*J
    return (Q+Q.transpose(0,2,1))/2

for split,theta in [('fit',th_fit),('test',th_test)]:
    x,u=make_points(theta)
    for sigma in SIGMAS:
        Q=learned_Q(x,sigma)
        err,defect,proxy,soft_dim,_=diagnostics(Q,u,x,theta,sigma,np.random.default_rng(SEED+13))
        for i in range(len(x)):
            records.append({'model':'learned_score','split':split,'index':i,'sigma':sigma,'tangent_error':err[i],'projector_defect':defect[i],'neighbor_proxy':proxy[i],'soft_dimension':soft_dim[i]})

## 5 — Compare actual geometric error vs. proxy ranking

**Fit-only selection:** choose \(\sigma\) minimizing the *mean neighbor-direction proxy* on the **40 fitting points**. Then inspect the **100 held-out points** using the known projector. No test-point true tangent information is used to select a scale.

Report paired differences \(e_T(\sigma=.342)-e_T(\sigma=.030)\) and a simple 95% paired bootstrap interval. The interval quantifies point-sampling uncertainty for one fixed trained model; it does **not** include training-seed variability.

In [ ]:
df=pd.DataFrame(records)
summary=df[df.split=='test'].groupby(['model','sigma']).agg(mean_error=('tangent_error','mean'),median_error=('tangent_error','median'),mean_proxy=('neighbor_proxy','mean'),mean_defect=('projector_defect','mean'),soft_dimension=('soft_dimension','mean')).reset_index()
print('\nHELD-OUT RESULTS');print(summary.to_string(index=False))
for model_name,g in df.groupby('model'):
    fit=g[g.split=='fit']
    test=g[g.split=='test']
    best_sigma=float(fit.groupby('sigma')['neighbor_proxy'].mean().idxmin())
    errors=test.pivot(index='index',columns='sigma',values='tangent_error')
    delta=errors[0.342]-errors[0.030]
    boot_rng=np.random.default_rng(SEED+40)
    draws=boot_rng.integers(0,len(delta),size=(2000,len(delta)))
    boot=delta.to_numpy()[draws].mean(axis=1)
    print('\n',model_name,'fit-only proxy-best sigma=',best_sigma,' heldout true-error change 0.342-0.030=',np.mean(delta),'95% paired bootstrap CI=',np.quantile(boot,[.025,.975]))
    for sigma,sg in test.groupby('sigma'):
        rho,p=spearmanr(sg['neighbor_proxy'],sg['tangent_error'])
        print('sigma',sigma,'proxy v true error spearman',rho,'p=',p)

## 6 — Visualize and persist the experiment

The **true tangent error** is the critical outcome, not whether a method produces the smallest proxy. The optional `soft_dimension` diagnostic, \(\sum_i(1-\mathrm{clip}(\lambda_i(Q),0,1))\), is **not** the same as the LHSD / Hill-filter dimension estimate used in your MNIST experiments; do not compare its value directly to those results.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,g in summary.groupby('model'):
    axes[0].plot(g['sigma'],g['mean_error'],marker='o',label=name)
    axes[1].plot(g['sigma'],g['mean_proxy'],marker='o',label=name)
axes[0].set(title='True tangent-projector error', xlabel='Gaussian noise sigma', ylabel='Held-out mean relative Frobenius error')
axes[1].set(title='Neighbor-chord proxy loss', xlabel='Gaussian noise sigma', ylabel='Held-out mean proxy')
for ax in axes:
    ax.legend();ax.grid(alpha=.2)
fig.tight_layout()
plt.show()
fig.savefig(OUT/'true_error_vs_proxy.png',dpi=180,bbox_inches='tight')
df.to_csv(OUT/'all_per_point_results.csv',index=False)
summary.to_csv(OUT/'heldout_summary.csv',index=False)

report={
 'experiment':'uniform circle in R8; true tangent dimension 1',
 'seed':SEED,'fit_points':N_FIT,'heldout_points':N_TEST,
 'sigmas':SIGMAS,'training_steps':TRAIN_STEPS,
 'per_model':{},
}
for name,g in df.groupby('model'):
    train=g[g.split=='fit'];heldout=g[g.split=='test']
    proxy_selected=float(train.groupby('sigma')['neighbor_proxy'].mean().idxmin())
    # Exact oracle is geometrically perfect at all scales; tiny floating-point
    # differences must not be misreported as a meaningful optimal sigma.
    mean_true=heldout.groupby('sigma')['tangent_error'].mean()
    all_numerically_exact=bool((mean_true<1e-5).all())
    true_best=None if all_numerically_exact else float(mean_true.idxmin())
    report['per_model'][name]={
       'fit_only_proxy_selected_sigma':proxy_selected,
       'heldout_lowest_true_error_sigma':true_best,
       'true_error_equivalent_across_scales':all_numerically_exact,
       'do_they_agree':None if all_numerically_exact else bool(np.isclose(proxy_selected,true_best)),
    }
(OUT/'experiment_summary.json').write_text(json.dumps(report,indent=2))
print('Saved checkpoint, CSV, plot, and JSON to:',OUT)
print(json.dumps(report,indent=2))

## 7 — Interpretation for your PhD prototype

**If oracle errors are ~0:** The spectral orientation extraction is implemented correctly for this tractable manifold. It does not prove neural score accuracy.

**If learned-score errors are small:** There is evidence that a learned score can recover true tangent directions in this controlled geometry.

**If the proxy selects a different sigma than the true-error minimum:** Our proxy-alignment repair is incomplete. A lower proxy is not a certificate of tangent accuracy, even with independent fitting and calibration.

**If the proxy and ground truth agree:** That supports the proxy only on the tested synthetic model/geometry; repeat on circles of different radii, spheres, nonuniform densities, higher ambient dimensions, and several training seeds before generalizing.

### Important follow-up

1. The current test **supplies known \(d=1\)** when extracting the tangent eigenvector. Next, evaluate whether your unsupervised **dimension estimator** recovers \(d=1\) and how its errors affect \(\hat P_T\).
2. Then compare the learned **manifold-walk directions** and their ability to remain on the synthetic manifold. On a circle, distance-to-manifold is \(\big|\|B^\top y\|-1\big|\) only if \(y\) is already in the containing plane; in general use \(\sqrt{(\|B^\top y\|-1)^2+\|y-BB^\top y\|^2}\).
3. Do **not** infer that a noise scale with the same numerical sigma is optimal for MNIST. Synthetic radius, ambient dimension, and score model quality all affect the optimal scale.
4. Only after that, transfer the testing protocol—not the synthetic-trained network—to your pretrained MNIST model.

---
# 8 — Cross-manifold extension (additive; original circle results unchanged)

**Purpose:** Generate independent `ellipse_per_point_results.csv`, `sphere_per_point_results.csv`, and `torus_per_point_results.csv` with the **same score-network family, finite-difference score Jacobian, noise grid and four predictor features** as your circle experiment. True tangent projectors are known analytically and used **only to compute evaluation labels**.

Key design choices:
- All datasets are embedded in the original $D=8$ ambient space. The ellipse has tangent dimension $d=1$; sphere and torus have $d=2$.
- `tangent_error` is $\|\hat P_T-P_T\|_F/\sqrt{d}$. This **exactly matches** your original circle `tangent_error` for $d=1$ and makes different dimensions comparable on a $[0,\sqrt2]$ scale.
- `neighbor_proxy` retains your original **normalized chord energy / random direction energy** formula. For $d=2$ we evaluate chords along each of the two intrinsic coordinate directions; the number of directions grows to cover the higher-dimensional tangent space.
- `projector_defect` and `soft_dimension` retain their original formulas.
- Ground truth (`T_true` or `tangent_error`) is **not** a predictor feature. The score model for each new manifold is independently trained.
- The original circle results are saved unchanged to `all_per_point_results.csv`; a separate `circle_per_point_results.csv` fixes reused fit/test index IDs for the transfer notebook.

**Interpretation caution:** One-dimensional versus two-dimensional data naturally changes some feature distributions, so a transfer failure can reflect shift in both intrinsic dimension and geometry—not necessarily failure of the reliability principle itself. The ellipse provides the closer, same-dimension test.

In [ ]:
# Analytical embeddings, true tangent bases and neighborhood sampling.
# Uses D, B, SIGMAS, N_FIT, N_TEST, SEED, DEVICE, OUT, EpsNetwork from original cells.
import numpy as np, pandas as pd, torch

# Keep the original circle plane, extend it by one orthonormal axis for surfaces.
_extension_rng = np.random.default_rng(SEED+101)
_vec = _extension_rng.normal(size=D)
_vec = _vec - B @ (B.T @ _vec)
_vec = _vec/np.linalg.norm(_vec)
B3 = np.column_stack([B, _vec])  # columns are orthonormal; shape D x 3
A_ELL, B_ELL = 1.30, 0.70
R_SPHERE = 1.0
R_MAJOR, R_MINOR = 1.20, 0.45
FAMILIES = ('ellipse','sphere','torus')
OFFSET = np.array([-0.14,-0.08,0.08,0.14],dtype=float)

def sample_family_params(family, n, rng):
    if family == 'ellipse':
        return rng.uniform(0, 2*np.pi, size=(n,1))
    if family == 'sphere':
        return np.column_stack([rng.uniform(0,2*np.pi,size=n), rng.uniform(-1,1,size=n)])
    if family == 'torus':
        return rng.uniform(0,2*np.pi,size=(n,2))
    raise ValueError(family)

def family_points_and_tangent(family, params):
    # Return x:(n,D), true orthonormal tangent rows:(n,d,D).
    p = np.asarray(params,dtype=float)
    if family=='ellipse':
        th=p[:,0]; c=np.cos(th); s=np.sin(th)
        local=np.column_stack([A_ELL*c, B_ELL*s])
        x=local@B.T
        v=(-A_ELL*s)[:,None]*B[:,0]+(B_ELL*c)[:,None]*B[:,1]
        return x, (v/np.linalg.norm(v,axis=1,keepdims=True))[:,None,:]
    if family=='sphere':
        th,z=p[:,0],p[:,1]
        a=np.sqrt(np.clip(1-z*z,1e-12,None))
        local=R_SPHERE*np.column_stack([a*np.cos(th),a*np.sin(th),z])
        t1=np.column_stack([-np.sin(th),np.cos(th),np.zeros_like(th)])
        t2=np.column_stack([-z*np.cos(th),-z*np.sin(th),a])
        x=local@B3.T
        T=np.stack([t1@B3.T,t2@B3.T],axis=1)
        return x,T
    if family=='torus':
        u,v=p[:,0],p[:,1]
        cv,sv,cu,su=np.cos(v),np.sin(v),np.cos(u),np.sin(u)
        radial=R_MAJOR+R_MINOR*cv
        local=np.column_stack([radial*cu,radial*su,R_MINOR*sv])
        # Coordinate derivative directions normalized to length 1.
        t1=np.column_stack([-su,cu,np.zeros_like(u)])
        t2=np.column_stack([-sv*cu,-sv*su,cv])
        return local@B3.T, np.stack([t1@B3.T,t2@B3.T],axis=1)
    raise ValueError(family)

def family_neighbor_chords(family,params,x,T):
    # Keep the original four offset magnitudes. For 2D, cover both tangent axes.
    n,d,_=T.shape
    neighbors=[]
    for j in range(d):
        for delta in OFFSET:
            if family=='ellipse':
                th=params[:,0]
                speed=np.sqrt((A_ELL*np.sin(th))**2+(B_ELL*np.cos(th))**2)
                shifted=params.copy(); shifted[:,0]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            elif family=='sphere':
                # Exact sphere great-circle step of arc length |delta|.
                x2=np.cos(delta/R_SPHERE)*x+(R_SPHERE*np.sin(delta/R_SPHERE))*T[:,j,:]
            elif family=='torus':
                u,v=params[:,0],params[:,1]
                speed=(R_MAJOR+R_MINOR*np.cos(v)) if j==0 else np.full(n,R_MINOR)
                shifted=params.copy(); shifted[:,j]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            vec=x2-x
            neighbors.append(vec/(np.linalg.norm(vec,axis=1,keepdims=True)+1e-12))
    return np.stack(neighbors,axis=1)

def generalized_diagnostics(Q,T,x,params,family,rng):
    # Same original proxies; projector error normalized by sqrt(true tangent dim).
    Q=0.5*(Q+np.swapaxes(Q,-1,-2))
    eig,V=np.linalg.eigh(Q)
    n,d,Dlocal=T.shape
    P_hat=V[:,:,:d] @ np.swapaxes(V[:,:,:d],-1,-2)
    P_true=np.swapaxes(T,-1,-2)@T
    overlap=np.einsum('nij,nij->n',P_hat,P_true)
    err=np.sqrt(np.maximum(0.0,2.0*(d-overlap)/d))
    defect=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
    nb=family_neighbor_chords(family,params,x,T)
    rand=rng.normal(size=(n,16,Dlocal))
    rand/=np.linalg.norm(rand,axis=2,keepdims=True)+1e-12
    nb_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,nb)**2,axis=-1),axis=1)
    rand_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,rand)**2,axis=-1),axis=1)
    proxy=np.minimum(1.0,nb_energy/(rand_energy+1e-12))
    soft_dim=np.sum(1-np.clip(eig,0,1),axis=1)
    return err,defect,proxy,soft_dim

# Sanity checks for analytic tangent projectors.
for fam in FAMILIES:
    ps=sample_family_params(fam,30,np.random.default_rng(SEED+222))
    x,t=family_points_and_tangent(fam,ps)
    should_identity=t@np.swapaxes(t,-1,-2)
    assert np.allclose(should_identity,np.eye(t.shape[1]),atol=1e-6),f'{fam}: invalid tangent basis'
    assert x.shape==(30,D)
    print(fam,'intrinsic_dim',t.shape[1],'ambient_dim',x.shape[1], 'tangent_basis_orthonormal',True)


In [ ]:
# Train an independent noise-conditioned score network per unseen manifold.
# Checkpoint caching permits re-running evaluation without retraining.
# EXT_STEPS can be increased if scores remain poorly trained.
EXT_STEPS=TRAIN_STEPS        # 2500 with original defaults; increase for ablations if needed.
FORCE_RETRAIN=False        # Set True when changing geometry or training hyperparameters.

def sample_clean_torch(fam,n):
    if fam=='ellipse':
        t=2*np.pi*torch.rand(n,device=DEVICE)
        loc=torch.stack([A_ELL*torch.cos(t),B_ELL*torch.sin(t)],dim=1)
        return loc @ torch.as_tensor(B.T,dtype=torch.float32,device=DEVICE)
    if fam=='sphere':
        t=2*np.pi*torch.rand(n,device=DEVICE)
        z=2*torch.rand(n,device=DEVICE)-1
        r=torch.sqrt(torch.clamp(1-z*z,min=0))
        loc=R_SPHERE*torch.stack([r*torch.cos(t),r*torch.sin(t),z],dim=1)
        return loc @ torch.as_tensor(B3.T,dtype=torch.float32,device=DEVICE)
    if fam=='torus':
        u=2*np.pi*torch.rand(n,device=DEVICE)
        v=2*np.pi*torch.rand(n,device=DEVICE)
        rr=R_MAJOR+R_MINOR*torch.cos(v)
        loc=torch.stack([rr*torch.cos(u),rr*torch.sin(u),R_MINOR*torch.sin(v)],dim=1)
        return loc @ torch.as_tensor(B3.T,dtype=torch.float32,device=DEVICE)
    raise ValueError(fam)

def train_family_model(fam):
    # Filename includes major setup info to avoid silently loading another geometry's weights.
    ckpt=OUT/f'{fam}_eps_R8_seed{SEED}_steps{EXT_STEPS}_v1.pt'
    torch.manual_seed(SEED+1000+FAMILIES.index(fam))
    net=EpsNetwork(D).to(DEVICE)
    if ckpt.exists() and not FORCE_RETRAIN:
        net.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
        print(f'{fam}: loaded checkpoint {ckpt.name}')
    else:
        opt=torch.optim.Adam(net.parameters(),lr=1e-3)
        sig_t=torch.tensor(SIGMAS,device=DEVICE,dtype=torch.float32)
        for it in range(EXT_STEPS):
            idx=torch.randint(len(SIGMAS),(BATCH,),device=DEVICE)
            sigma=sig_t[idx,None]
            clean=sample_clean_torch(fam,BATCH)
            eps=torch.randn_like(clean)
            predicted=net(clean+sigma*eps,sigma)
            loss=(predicted-eps).square().mean()
            opt.zero_grad();loss.backward();opt.step()
            if it==0 or (it+1)%500==0 or it+1==EXT_STEPS:
                print(f'{fam:>7} | step {it+1:>5}/{EXT_STEPS} | noise-prediction loss={loss.item():.4f}')
        torch.save(net.state_dict(),ckpt)
        print(f'{fam}: checkpoint saved: {ckpt}')
    return net.eval()

@torch.no_grad()
def learned_Q_for(net,x,sigma,h=0.001):
    # Identical centered finite differences and Q = sigma * J_eps from original cell 10.
    X=np.asarray(x,dtype=np.float32)
    cols=[]
    for j in range(D):
        plus=X.copy();minus=X.copy();plus[:,j]+=h;minus[:,j]-=h
        a=net(torch.from_numpy(plus).to(DEVICE),float(sigma)).cpu().numpy()
        b=net(torch.from_numpy(minus).to(DEVICE),float(sigma)).cpu().numpy()
        cols.append((a-b)/(2*h))
    J=np.stack(cols,axis=-1)
    Q=float(sigma)*J
    return 0.5*(Q+np.swapaxes(Q,-1,-2))


In [ ]:
# Generate three transfer-ready CSVs: analytic ground truth is used only for LABEL generation.
family_tables={}
for fam in FAMILIES:
    print('\n'+'='*70+'\nGENERATING '+fam.upper()+'\n'+'='*70)
    net=train_family_model(fam)
    rows=[]
    for k,(split,n) in enumerate([('fit',N_FIT),('test',N_TEST)]):
        params=sample_family_params(fam,n,np.random.default_rng(SEED+300+10*FAMILIES.index(fam)+k))
        x,T=family_points_and_tangent(fam,params)
        for sig in SIGMAS:
            Q=learned_Q_for(net,x,sig)
            err,defect,proxy,softdim=generalized_diagnostics(
                Q,T,x,params,fam,np.random.default_rng(SEED+13))
            # Crucial: fit and test IDs must be disjoint for group-split validation!
            for i in range(n):
                rows.append(dict(model='learned_score',split=split,
                    index=(i if split=='fit' else N_FIT+i),sigma=float(sig),
                    tangent_error=float(err[i]),projector_defect=float(defect[i]),
                    neighbor_proxy=float(proxy[i]),soft_dimension=float(softdim[i])))
    tab=pd.DataFrame(rows)
    assert not tab[['tangent_error','projector_defect','neighbor_proxy','soft_dimension']].isna().any().any()
    assert not tab.groupby('index')['split'].nunique().gt(1).any()
    path=OUT/f'{fam}_per_point_results.csv'
    tab.to_csv(path,index=False)
    family_tables[fam]=tab
    print('Saved:',path,'| learned score rows:',len(tab))
    print(tab[tab.split=='test'].groupby('sigma')['tangent_error'].agg(['mean','median']).round(4))
    del net
    if DEVICE=='cuda': torch.cuda.empty_cache()


In [ ]:
# Prepare the circle CSV for the existing Cross_Manifold_Transfer_Test_Colab notebook.
# Original code reuses indices 0..39 in fit and 0..99 in test; transfer checks need disjoint point IDs.
# Do not overwrite the original all_per_point_results.csv.
original_csv = OUT/'all_per_point_results.csv'
if not original_csv.exists():
    raise FileNotFoundError(f'Run the original experiment through Cell 14 first: {original_csv}')
circle_transfer=pd.read_csv(original_csv).copy()
assert {'model','split','index','sigma','tangent_error','projector_defect','neighbor_proxy','soft_dimension'} <= set(circle_transfer)
assert set(circle_transfer['split'])=={'fit','test'}
# Preserve original point identities within each split; make fit/test IDs disjoint across both model types.
offset=int(circle_transfer.loc[circle_transfer.split=='fit','index'].max())+1
circle_transfer.loc[circle_transfer.split=='test','index'] += offset
assert not circle_transfer.groupby('index')['split'].nunique().gt(1).any()
path=OUT/'circle_per_point_results.csv'
circle_transfer.to_csv(path,index=False)
print('Saved transfer-ready circle:',path)

# Verify the expected feature schema and common scale grid.
required={'model','split','index','sigma','tangent_error','projector_defect','neighbor_proxy','soft_dimension'}
for fam,file in [('circle',path)]+[(f,OUT/f'{f}_per_point_results.csv') for f in FAMILIES]:
    data=pd.read_csv(file)
    sub=data[data.model=='learned_score']
    assert not (required-set(data.columns)),(fam,required-set(data.columns))
    assert set(sub.sigma.round(6))==set(np.round(SIGMAS,6)),fam
    assert sub.groupby(['split','index']).size().eq(len(SIGMAS)).all(),fam
    print(f'{fam:8s}: {len(sub):4d} learned-score rows, '
          f'{sub.loc[sub.split=="test","index"].nunique():3d} test points, '
          f'mean test tangent error={sub[sub.split=="test"].tangent_error.mean():.5f}')
print('\nALL FOUR CSVs READY at:',OUT)
print('In Cross_Manifold_Transfer_Test_Colab.ipynb, set:')
print('USE_COLAB_UPLOAD = False')
print(f'DATA_DIR = Path({str(OUT)!r})')


## 9 — Run the frozen transfer experiment

1. Open **Cross_Manifold_Transfer_Test_Colab.ipynb**.
2. In its input cell set `USE_COLAB_UPLOAD = False` and `DATA_DIR = Path('/content/drive/MyDrive/synthetic_tangent_ground_truth')` (or the exact `OUT` path printed above).
3. Run the cross-manifold notebook from the top. It discovers `circle_per_point_results.csv`, `ellipse_per_point_results.csv`, `sphere_per_point_results.csv` and `torus_per_point_results.csv` automatically.
4. Compare ellipse first (same intrinsic dimension as circle), then sphere and torus (different intrinsic dimension). Inspect `transfer_summary.csv`, `paired_improvements.csv`, the simultaneous coverage of calibrated bounds and risk–coverage plots.

**Do not use any unseen family's `tangent_error` to refit the circle predictor.** Ground truth for the unseen families is for **evaluation only**.

**Scientific limits:** These tests assess cross-family transfer on synthetic manifolds with known tangents. Strong transfer would be evidence of robustness to some geometry shifts, *not proof of universal geometry principles or MNIST performance*. Poor transfer might reflect predictor feature shift, dimension shift, score-model quality or geometry shift; use feature shift diagnostics to disentangle these.